In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from helpers import get_table, get_bronze
from functools import reduce

print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("silver_processing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/04/27 18:37:07 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 18:37:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-07f6aec0-65e5-45ea-abde-66fe92a87b5d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 537ms :: artifacts dl 44ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# Support tickets silver

Grain: one row per `ticket_id`.

Processing steps:
+ Read Bronze `support_tickets`.
+ Cast ticket IDs, user IDs, category, description, timestamps, and lineage fields.
+ Normalize category and clean text for downstream NLP.
+ Quarantine null keys, malformed timestamps, invalid categories, meaningless descriptions, and orphan users.
+ Deduplicate by `ticket_id`.
+ Derive description length and ticket date parts.
+ Upsert into Delta path `./silver/support_tickets`.

## Read Bronze support_tickets data

In [3]:
support_tickets_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/support_tickets"
df_bronze_support_tickets = get_bronze(support_tickets_bronze, spark=spark).drop("id", "source_identifier", "batch_id")
df_bronze_support_tickets.show(5, truncate=False)

[Stage 8:>                                                          (0 + 1) / 1]

+---------+-------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|ticket_id|user_id|category |description                                                                                                                                                                                                                                                                                                                                             |created_at         |ingest_time               |
+---------+-------+---------+-------------------------------------------------------------------------------------------------------------------------------

In [4]:
df_bronze_support_tickets.printSchema()
df_bronze_support_tickets.count()

root
 |-- ticket_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- ingest_time: timestamp (nullable = true)



22255

## Cast and normalize fields

In [6]:
df1 = (
    df_bronze_support_tickets
    .select(
        F.col("ticket_id").cast("bigint").alias("ticket_id"),
        F.col("user_id").cast("bigint").alias("user_id"),
        F.col("category").cast("string").alias("category"),
        F.col("description").cast("string").alias("description"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        F.col("ingest_time").cast("timestamp").alias("ingest_time"),
    ))

df1.show(5, truncate=False)

+---------+-------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|ticket_id|user_id|category |description                                                                                                                                                                                                                                                                                                                                             |created_at         |ingest_time               |
+---------+-------+---------+-------------------------------------------------------------------------------------------------------------------------------

## Validate required fields and domains

In [7]:
valid_ticket_categories = ["billing", "technical", "cancellation", "feature_request"]
meaningless_descriptions = ["", "n/a", "na", "none", "null", "test", "no description"]

df_tickets_validated = (
    df1
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(F.col("ticket_id").isNull(), F.lit("ticket_id is null")),
            F.when(F.col("user_id").isNull(), F.lit("user_id is null")),
            F.when(F.col("created_at").isNull(), F.lit("created_at is null")),
            F.when(F.col("category").isNull() | ~F.col("category").isin(valid_ticket_categories), F.lit("invalid category")),
        )
    )
    .drop("description_lower")
)

df2 = df_tickets_validated.filter(F.col("validation_error") == "")
df2_quarantine = df_tickets_validated.filter(F.col("validation_error") != "")

df2.count(), df2_quarantine.count()

(22255, 0)

## Deduplicate at ticket grain

In [8]:
w = Window.partitionBy("ticket_id", "created_at").orderBy(F.col("ingest_time").desc())

df_tickets_ranked = df2.withColumn("rn", F.row_number().over(w))

df3 = df_tickets_ranked.filter(F.col("rn") == 1).drop("rn", "validation_error")
df3_quarantine = (
    df_tickets_ranked
    .filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate ticket_id"))
)

df3.count(), df3_quarantine.count()

(22255, 0)

## Validate user_id referential integrity

In [9]:
silver_users_path = "./silver/users"
df_silver_users = (
    spark.read
    .format("delta")
    .load(silver_users_path)
)

df_users_ref = df_silver_users.select(F.col("user_id").cast("bigint").alias("user_id")).dropDuplicates()

df4 = (
    df3.alias("t")
    .join(df_users_ref.alias("u"), on="user_id", how="inner")
)

df4_quarantine = (
    df2.alias("t")
    .join(df_users_ref.alias("u"), on="user_id", how="left_anti")
    .withColumn("validation_error", F.lit("user_id not found in silver users"))
)

df4.count(), df4_quarantine.count()

(22255, 0)

## Get all quarantine tables

In [10]:
quarantine_dfs = [
    df2_quarantine,
    df3_quarantine,
    df4_quarantine,
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)

df_quarantine_all.count()

0

In [11]:
df_support_tickets_silver = df4

## Upsert strategy

In [12]:
silver_path = "./silver/support_tickets"

silver_cols = df_support_tickets_silver.columns

df_upsert = df_support_tickets_silver.select(*silver_cols)

w = Window.partitionBy("ticket_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())
df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("Update existing table")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(df_upsert.alias("source"), "target.ticket_id = source.ticket_id")
        .whenMatchedUpdate(set={c: f"source.{c}" for c in silver_cols if c != "ticket_id"})
        .whenNotMatchedInsert(values={c: f"source.{c}" for c in silver_cols})
        .execute()
    )
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table does not exist yet


In [13]:
df_support_tickets_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_support_tickets_silver_read.show(5, truncate=False)
df_support_tickets_silver_read.count()

+-------+---------+---------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|user_id|ticket_id|category       |description                                                                                                                                                                                                                                                                                                                                             |created_at         |ingest_time               |
+-------+---------+---------------+-------------------------------------------------------------------------------------------------------------

22255

In [14]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused